# # Experimental Models - Rent Price Prediction

In [3]:
from dotenv import load_dotenv
from mlModels.linearRegression.rent.logPrice.data import getData, getLinearRegressionData
from mlModels.kmeans.rent.runCluster import runCluster
import os

CV = True
CV_VAL = 5

os.chdir("/home/florian/Desktop/immopreis-regression")

load_dotenv("database/.env")

df_features = getData()
cluster_dict = runCluster()

/home/florian/Desktop/immopreis-regression/mlModels/linearRegression/rent/logPrice/data.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_features = pd.read_sql("SELECT * FROM rent_features", conn)
/home/florian/Desktop/immopreis-regression/mlModels/linearRegression/rent/logPrice/data.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_listings = pd.read_sql("SELECT * FROM listings", conn)
/home/florian/Desktop/immopreis-regression/mlModels/kmeans/rent/data.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = 


-------Starting K-Means Clustering locations-------


# Linear Regression

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

for k, df_cluster in cluster_dict.items():
    df = df_features.merge(df_cluster, on="id", how="inner")
    df_house_X, df_house_y, df_apt_X, df_apt_y = getLinearRegressionData(df)

    X_train, X_test, y_train, y_test = train_test_split(df_apt_X, df_apt_y, test_size=0.2, random_state=42)

    model = LinearRegression()

    if CV:
        scores = cross_val_score(model, X_train, y_train, cv=CV_VAL, scoring="r2", verbose=True)
        print(f"\n-------Cross Validation Scores for K={k}-------")
        print(f"CV R2 Scores: {scores}")
        print(f"Mean R2 Score: {scores.mean():.4f}")
        print(f"Standard Deviation: {scores.std():.4f}")
        print()

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"-------Linear Regression Results for K={k}-------")
    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2:   {r2:.4f}")
    print()


-------Cross Validation Scores for K=3-------
CV R2 Scores: [0.50336298 0.53615484 0.75087096 0.68863146 0.63535176]
Mean R2 Score: 0.6229
Standard Deviation: 0.0924

-------Linear Regression Results for K=3-------
MAE:  0.1598
RMSE: 0.2079
R2:   0.5971


-------Cross Validation Scores for K=4-------
CV R2 Scores: [0.51080576 0.53961234 0.75713975 0.67131462 0.62770179]
Mean R2 Score: 0.6213
Standard Deviation: 0.0893

-------Linear Regression Results for K=4-------
MAE:  0.1630
RMSE: 0.2092
R2:   0.5919


-------Cross Validation Scores for K=14-------
CV R2 Scores: [0.53760746 0.59126696 0.73786213 0.75462396 0.67382881]
Mean R2 Score: 0.6590
Standard Deviation: 0.0836

-------Linear Regression Results for K=14-------
MAE:  0.1527
RMSE: 0.2011
R2:   0.6230



[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.1s finished
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.0s finished


# Lasso

In [6]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

for k, df_cluster in cluster_dict.items():
    df = df_features.merge(df_cluster, on="id", how="inner")
    df_house_X, df_house_y, df_apt_X, df_apt_y = getLinearRegressionData(df)

    model = Lasso(alpha=0.1)

    X_train, X_test, y_train, y_test = train_test_split(df_apt_X, df_apt_y, test_size=0.2, random_state=42)

    if CV:
        scores = cross_val_score(model, X_train, y_train, cv=CV_VAL, scoring="r2", verbose=True)
        print(f"\n-------Cross Validation Scores for K={k}-------")
        print(f"CV R2 Scores: {scores}")
        print(f"Mean R2 Score: {scores.mean():.4f}")
        print(f"Standard Deviation: {scores.std():.4f}")
        print()

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print("-------Lasso Regression Results-------")
    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2:   {r2:.4f}")




-------Cross Validation Scores for K=3-------
CV R2 Scores: [0.09301247 0.19629116 0.1094669  0.21389861 0.10247181]
Mean R2 Score: 0.1430
Standard Deviation: 0.0512

-------Lasso Regression Results-------
MAE:  0.2403
RMSE: 0.2940
R2:   0.1944

-------Cross Validation Scores for K=4-------
CV R2 Scores: [0.09301247 0.19629116 0.1094669  0.21389861 0.10247181]
Mean R2 Score: 0.1430
Standard Deviation: 0.0512

-------Lasso Regression Results-------
MAE:  0.2403
RMSE: 0.2940
R2:   0.1944

-------Cross Validation Scores for K=14-------
CV R2 Scores: [0.09301247 0.19629116 0.1094669  0.21389861 0.10247181]
Mean R2 Score: 0.1430
Standard Deviation: 0.0512

-------Lasso Regression Results-------
MAE:  0.2403
RMSE: 0.2940
R2:   0.1944


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.0s finished


# Ridge

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
import numpy as np

for k, df_cluster in cluster_dict.items():
    df = df_features.merge(df_cluster, on="id", how="inner")

    df_house_X, df_house_y, df_apt_X, df_apt_y = getLinearRegressionData(df)
    X_train, X_test, y_train, y_test = train_test_split(df_apt_X, df_apt_y, test_size=0.2, random_state=42)

    model = Ridge(alpha=0.1)

    if CV:
        scores = cross_val_score(model, X_train, y_train, cv=CV_VAL, scoring="r2", verbose=True)
        print(f"\n-------Cross Validation Scores for K={k}-------")
        print(f"CV R2 Scores: {scores}")
        print(f"Mean R2 Score: {scores.mean():.4f}")
        print(f"Standard Deviation: {scores.std():.4f}")
        print()

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"-------Ridge Regression Results for K={k}-------")
    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2:   {r2:.4f}")
    print()


-------Cross Validation Scores for K=3-------
CV R2 Scores: [0.50549335 0.56113448 0.75311319 0.69052302 0.63543814]
Mean R2 Score: 0.6291
Standard Deviation: 0.0884

-------Ridge Regression Results for K=3-------
MAE:  0.1593
RMSE: 0.2071
R2:   0.6002


-------Cross Validation Scores for K=4-------
CV R2 Scores: [0.51302436 0.56526769 0.7598939  0.6761636  0.62736346]
Mean R2 Score: 0.6283
Standard Deviation: 0.0859

-------Ridge Regression Results for K=4-------
MAE:  0.1628
RMSE: 0.2087
R2:   0.5942


-------Cross Validation Scores for K=14-------
CV R2 Scores: [0.55016691 0.59521714 0.7434018  0.75282579 0.67336947]
Mean R2 Score: 0.6630
Standard Deviation: 0.0800

-------Ridge Regression Results for K=14-------
MAE:  0.1518
RMSE: 0.1981
R2:   0.6341



[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.0s finished


# Elastic Net

In [ ]:
from sklearn.linear_model import ElasticNet

model = ElasticNet(alpha=0.1, l1_ratio=0.5)
model.fit(df_apt_X, df_apt_y)